In [21]:
# Importar librerías
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [6]:
# 1. Configurar rutas
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / 'data'
ruta_datos = DATA_DIR / 'Dataset_Smart_Farming_base.csv'
# 2. Cargar datos EDA
df = pd.read_csv(ruta_datos)

In [7]:
# 1. Reemplazar cadenas vacías por NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

,farm_id,region,crop_type,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,irrigation_type,...,yield_kg_per_hectare,sensor_id,timestamp,latitude,longitude,NDVI_index,crop_disease_status,N,P,K
0,FARM0001,North India,Wheat,35.95,5.99,17.456436,75.62,76.406275,7.27,NaN,...,4408.07,SNS-2026-0001,2024-03-19,14.970941,82.997689,0.63,Mild,11,60,23
1,FARM0002,South USA,Soybean,19.74,7.24,30.275671,89.91,60.899686,5.67,Sprinkler,...,5389.98,SNS-2026-0002,2024-04-21,16.613022,70.869009,0.58,NaN,136,36,20
2,FARM0003,South USA,Wheat,29.32,7.16,27.453712,265.43,68.954600,8.23,Drip,...,2931.16,SNS-2026-0003,2024-02-28,19.503156,79.068206,0.80,Mild,23,45,23
3,FARM0004,Central USA,Maize,17.33,6.03,34.033155,212.01,69.238224,5.03,Sprinkler,...,4227.80,SNS-2026-0004,2024-05-14,31.071298,85.519998,0.44,NaN,113,6,52
4,FARM0005,Central USA,Cotton,19.37,5.92,33.685737,269.09,54.595901,7.93,NaN,...,4979.96,SNS-2026-0005,2024-04-13,16.568540,81.691720,0.84,Severe,101,92,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,FARM0496,Central USA,Rice,42.85,6.70,31.073145,52.35,80.614607,7.25,Manual,...,4251.40,SNS-2026-2496,2024-05-08,30.386623,76.147700,0.59,Mild,58,73,16
2496,FARM0497,North India,Soybean,34.22,6.75,17.493631,256.23,44.890922,5.78,NaN,...,3708.54,SNS-2026-2497,2024-01-19,18.832748,75.736924,0.85,Severe,129,43,16
2497,FARM0498,North India,Cotton,15.93,5.72,17.553392,288.96,58.852243,7.69,Drip,...,2604.41,SNS-2026-2498,2024-04-20,23.262016,81.992230,0.71,Mild,105,74,45
2498,FARM0499,Central USA,Soybean,38.61,6.20,16.995694,279.06,72.951453,9.60,Drip,...,2586.36,SNS-2026-2499,2024-03-02,19.764989,84.426869,0.77,Severe,126,37,21


In [9]:
#  Eliminar columnas irrelevantes para la predicción
columnas_irrelevantes = [
    'farm_id', 'sensor_id', 'timestamp',
    'sowing_date', 'harvest_date',
    'latitude', 'longitude'
]
df_limpio = df.drop(columns=columnas_irrelevantes)

In [11]:
# Selección de características
# X: Todas las columnas excepto lo que queremos predecir
X = df_limpio.drop(columns=['crop_disease_status'])
# y: Lo que queremos predecir (El Guardián)
y = df_limpio['crop_disease_status']

In [15]:
# Filtro de seguridad: Forzamos la eliminación de filas sin etiqueta de enfermedad
df_limpio = df_limpio.dropna(subset=['crop_disease_status'])

# Filtro de seguridad: Forzamos el relleno del tipo de riego para que X no tenga nulos
df_limpio['irrigation_type'] = df_limpio['irrigation_type'].fillna('Desconocido')

# Separamos X (sensores/clima) e y (etiqueta a predecir)
X = df_limpio.drop(columns=['crop_disease_status'])
y = df_limpio['crop_disease_status']

In [16]:
# Partición de datos: Separamos 70% para estudiar y 30% para el examen final
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [17]:
# Categorización de variables
variables_categoricas = ['region', 'crop_type', 'irrigation_type', 'fertilizer_type']
# Automáticamente toma el resto como numéricas
variables_numericas = [col for col in X.columns if col not in variables_categoricas]

In [22]:
# Transformador (StandardScaler + OneHotEncoder
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), variables_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), variables_categoricas)
])

In [24]:
# Aplicamos la matemática al conjunto de entrenamiento y prueba
X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)
print(" Texto convertido a números (One-Hot) y sensores estandarizados.")

 Texto convertido a números (One-Hot) y sensores estandarizados.


In [26]:
# Muestreo de datos: Para balancear clases, aplicamos sobremuestreo a la clase minoritaria
from imblearn.over_sampling import SMOTE
print(" Clases ANTES de SMOTE en entrenamiento:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, y_train)

print("Clases DESPUÉS de SMOTE en entrenamiento:")
print(y_resampled.value_counts())

 Clases ANTES de SMOTE en entrenamiento:
crop_disease_status
Severe      466
Mild        437
Moderate    392
Name: count, dtype: int64
Clases DESPUÉS de SMOTE en entrenamiento:
crop_disease_status
Mild        466
Moderate    466
Severe      466
Name: count, dtype: int64
